# Quality of Life and Happiness Analysis

### Abstract

тема проекта - как внешние факторы влияют на кволити оф лайф и хапинесс скор в зависимости от страны на протяжении не сколкьких лет

# World Happiness Dataset

### Dataset Description


Dataset source:

https://www.kaggle.com/datasets/hari31416/world-happiness-report/data
The datasets belong to the field of social and economic statistics.

They contain annual data for multiple countries from 2015 to 2022 and include indicators such as economic conditions (GDP per capita), social support (Family), healthy life expectancy (Health), freedom to make life choices (Freedom), generosity, trust in institutions, and overall happiness scores.

For further analysis, the datasets will be merged into a single table using the additional **Year** column, which allows matching records from different years and enables time-series analysis.

In [ ]:
import pandas as pd

In [ ]:
tables = []
years = range(2015, 2023)

for year in years:
    table = pd.read_csv(f"../data_sets/{year}.csv")
    table["Year"] = year
    tables.append(table)


Before merging the datasets, it is needed to examine their structure and verify whether the columns are consistent across all years.

In [ ]:
reference_year = tables[0]


for i in range(len(tables)):
    if tables[i].columns.equals(reference_year.columns):
        print(f"{years[i]}: equal")
    else:
        print(f"{years[i]}: {set(tables[i].columns)^set(reference_year.columns)}")

The comparison of column names showed that the datasets do not have completely identical structures: the datasets for 2018 and 2019 do not contain the Dystopia column, while all other years include it.

Since the purpose of this project is to analyze variables that are available for every year, the datasets will be combined using only the columns that are present in all tables.

In [ ]:
all_data = pd.concat(tables, axis = 0, join = 'inner', ignore_index = True)

In [ ]:
all_data.head()

In [ ]:
all_data.shape

In [ ]:
all_data.info()

The merged dataset contains 1185 rows and 11 fields. The fields are represented by the following data types:
- 2 categorical fields (object): Country, Region
- 2 integer numerical fields (int64): Happiness Rank, Year
- 7 continuous numerical fields (float64): Happiness Score, Economy, Family, Health, Freedom, Generosity, Trust

In [ ]:
all_data.isna().sum()

In [ ]:
all_data.duplicated().sum()

The dataset is generally of high quality.

* No duplicate rows were found in the dataset.
* Only one missing value was detected in the `Trust` column, while all other fields were complete.
* The data types of all columns were examined and found to be appropriate for their contents: categorical variables are stored as `object`, numerical indicators as `float64`, and ranking and year values as `int64`. No inconsistent values or data type issues were
 identified yet.

 Therefore, the dataset can be considered clean and suitable for further analysis.


### Descriptive statistics

In [ ]:
statistics_h = pd.DataFrame ({

    "Mean":[
    all_data["Happiness Score"].mean(),
    all_data["Economy"].mean(),
    all_data["Family"].mean(),
    all_data["Health"].mean(),
    all_data["Freedom"].mean(),
    all_data["Generosity"].mean(),
    all_data["Trust"].mean()],

    "Median": [
    all_data["Happiness Score"].median(),
    all_data["Economy"].median(),
    all_data["Family"].median(),
    all_data["Health"].median(),
    all_data["Freedom"].median(),
    all_data["Generosity"].median(),
    all_data["Trust"].median()],

    "Standard Deviation": [
    all_data["Happiness Score"].std(),
    all_data["Economy"].std(),
    all_data["Family"].std(),
    all_data["Health"].std(),
    all_data["Freedom"].std(),
    all_data["Generosity"].std(),
    all_data["Trust"].std()],

    "Minimum": [
    all_data["Happiness Score"].min(),
    all_data["Economy"].min(),
    all_data["Family"].min(),
    all_data["Health"].min(),
    all_data["Freedom"].min(),
    all_data["Generosity"].min(),
    all_data["Trust"].min()],

    "Maximum": [
    all_data["Happiness Score"].max(),
    all_data["Economy"].max(),
    all_data["Family"].max(),
    all_data["Health"].max(),
    all_data["Freedom"].max(),
    all_data["Generosity"].max(),
    all_data["Trust"].max()],
},

index = ["The happiness score on a scale from 0 to 10",
         "The GDP per capita index", "Social Support",
         "Healthy Life Expectancy",
         "Freedom to make life choices",
         "Generosity in the community",
         "Trust in government"]
)

statistics_h

The descriptive statistics provide a general overview of the dataset and the distribution of its main numerical variables. Most indicators have reasonable ranges and average values that correspond to the expected scale of the World Happiness Report data.

However, the summary statistics also revealed potential inconsistencies in some variables. In particular, certain indicators show unusually large differences between their average values, medians, and maximum values, which may indicate that different measurement scales were used across parts of the dataset. These discrepancies will be investigated in more detail during the Data Cleanup stage, where the data will be checked for inconsistencies and, if necessary, standardized before further analysis.


### Data cleanup

During the initial exploration of the dataset, several data quality issues were identified:

* First, one missing value was found in the `Trust` column, while all other fields were complete.
* Second, the descriptive statistics revealed unusually large standard deviations and extreme values in the `Health` and `Economy` indicators.

These results suggest the presence of inconsistencies in the data, possibly caused by differences in measurement scales across years.

To ensure the reliability of further analysis, these issues will be investigated and addressed in the Data Cleanup stage. Missing values will be handled appropriately, and the identified inconsistencies will be examined and corrected where necessary.


In [ ]:
all_data = all_data.dropna()
all_data.isna().sum()

In [ ]:
all_data.groupby("Year")["Health"].describe()

In [ ]:
all_data.groupby("Year")["Economy"].describe()

In [ ]:
all_data = all_data.drop(columns=["Health", "Economy"])
all_data.columns

In [ ]:
all_data.groupby("Country")["Year"].nunique().value_counts().sort_index()

Some countries have data available for only a limited number of years. To improve the reliability and comparability of the analysis, only countries with observations for at least 50% of the study period (4 or more years out of 8) were retained. This approach reduces the impact of countries with insufficient data while preserving a substantial portion of the dataset.

In [ ]:
years_countries = all_data.groupby("Country")["Year"].nunique()
years_countries.value_counts().sort_index()

In [ ]:
good_countries = years_countries[years_countries == 8].index
all_data = all_data[all_data["Country"].isin(good_countries)]
all_data.head()

During the data cleaning process, one row containing a missing value in the Trust column was removed.

Further investigation revealed that the Health and Economy indicators were reported on substantially different scales in some years. Since no reliable conversion method was available and these variables were not essential for the subsequent analysis, they were removed from the dataset to ensure consistency.

### Plots

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(9,7))

sns.histplot(
    data=all_data,
    x="Generosity",
    bins=20,
    kde=True,
    color = "#FF69B4"
)

plt.title("Distribution of Generosity")
plt.xlabel("Generosity")
plt.ylabel("Count")

plt.show()

##### The histogram shows the distribution of the Generosity indicator across all countries and years included in the dataset.
 The distribution is right-skewed, with most values concentrated between -0.2 and 0.4, and a long tail extending toward higher values. This suggests that while most countries report relatively low or moderate levels of generosity, a smaller number of countries stand out with noticeably higher scores.

In [ ]:
plt.figure(figsize=(9,7))

sns.histplot(
    data=all_data,
    x="Family",
    bins=20,
    kde=True,
    color="#A8DADC"
)

plt.title("Distribution of Social Support")
plt.xlabel("Social Support")
plt.ylabel("Count")

plt.show()

The histogram shows the distribution of the Social Support indicator across all countries and years in the dataset. Most observations are concentrated between 0.8 and 1.3, indicating that moderate to high levels of social support are common among countries. Very low values occur relatively rarely, while the distribution suggests that social support is generally well developed in most observations included in the dataset.


In [ ]:
mean_happiness = (
    all_data.groupby("Year")["Happiness Score"]
    .mean()
    .reset_index()
)
plt.figure(figsize=(8,5))

sns.lineplot(
    data=mean_happiness,
    x="Year",
    y="Happiness Score",
    color = "pink",
    marker="o",
    linewidth=3
)

plt.title("Average Happiness Score by Year")
plt.xlabel("Year")
plt.ylabel("Average Happiness Score")

plt.show()

##### The line plot displays the trend in average Happiness Score from 2015 to 2022.
 The average score remains relatively stable, fluctuating between approximately 5.4 and 5.5 from 2015 to 2019. A noticeable drop occurs in 2020, falling below 5.2. In 2021–2022, the values show a partial recovery.

# Quality of Life Dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv("../data_sets/quality_of_life_indices_by_country.csv")

### Dataset description

Dataset source:

https://www.kaggle.com/datasets/marcelobatalhah/quality-of-life-index-by-country/data

The dataset belongs to the socio-economic domain and contains quality of life indicators for different countries between 2015 and 2024. The dataset describes different aspects of living conditions and well-being across countries.

In [ ]:
data.head()

In [ ]:
data.shape

The dataset contains 1495 rows and 12 columns.

There is one categorical field (Country), one ranking field (Rank), one time-related field (Year), and nine numerical indicators describing different aspects of quality of life.

These indicators include the overall Quality of Life Index, Purchasing Power Index, Safety Index, Health Care Index, Cost of Living Index, Property Price to Income Ratio, Traffic Commute Time Index, Pollution Index, and Climate Index.

In [ ]:
data.info()

According to the data types, most variables are numerical. However, the Year and Climate Index columns are stored as strings, indicating potential formatting issues that require further inspection.

In [ ]:
data.isna().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
print("Non-numeric Climate Index values:", (data["Climate Index"] == "-").sum())

In [ ]:
data["Year"].unique()

The dataset is generally of good quality. No explicit missing values or duplicate rows were detected.

However, some inconsistencies were found. The Year column contains values such as "2015/2", while the Climate Index column contains 143 non-numeric entries represented by "-". Because of these inconsistencies, both columns are stored as strings instead of numeric data types. These issues will be corrected during the data cleaning stage.

### Descriptive statistics

In [ ]:
statistics = pd.DataFrame({
    "Mean":[data["Quality of Life Index"].mean(),
            data["Purchasing Power Index"].mean(),
            data["Safety Index"].mean(),
            data["Health Care Index"].mean(),
            data["Cost of Living Index"].mean()],

    "Median":[
        data["Quality of Life Index"].median(),
        data["Purchasing Power Index"].median(),
        data["Safety Index"].median(),
        data["Health Care Index"].median(),
        data["Cost of Living Index"].median()],

    "Standard Deviation":
        [
        data["Quality of Life Index"].std(),
        data["Purchasing Power Index"].std(),
        data["Safety Index"].std(),
        data["Health Care Index"].std(),
        data["Cost of Living Index"].std()
        ],

    "Minimum":
        [
        data["Quality of Life Index"].min(),
        data["Purchasing Power Index"].min(),
        data["Safety Index"].min(),
        data["Health Care Index"].min(),
        data["Cost of Living Index"].min()
        ],
    "Maximum":
        [
        data["Quality of Life Index"].max(),
        data["Purchasing Power Index"].max(),
        data["Safety Index"].max(),
        data["Health Care Index"].max(),
        data["Cost of Living Index"].max()
        ]
},
    index=["Quality of Life Index",
           "Purchasing Power Index",
           "Safety Index",
           "Health Care Index",
           "Cost of Living Index"]
)

statistics.round(2)

Among the selected indicators, the Quality of Life Index has the highest variability, while the Health Care Index shows the lowest variability.

The Quality of Life Index also has the widest range of values, indicating substantial differences in living conditions across countries. The mean and median values are relatively close for most indicators, suggesting that the distributions are not strongly skewed.


### Data cleanup

In [ ]:
clean_data = data.drop(columns = "Climate Index")
clean_data.head()

Since the Climate Index column contains “-” for all observations from 2015, removing these values would result in losing an entire year of data. Therefore, the Climate Index column was removed from the dataset. Additionally, this indicator was not intended to be used in the correlation analysis or visualizations performed later in the project.

In [ ]:
clean_data["Year"] = clean_data["Year"].str.replace("/2", "")
clean_data["Year"] = clean_data["Year"].astype(int)

display(clean_data["Year"].unique())
display(clean_data.dtypes)

In [ ]:
clean_data = clean_data.groupby(["Country","Year"], as_index=False).mean()
clean_data.head(15)

The dataset contains semi-annual observations marked with the “/2” suffix. Since this project focuses on yearly analysis, the suffix was removed from the Year column and the values were converted to integers. Then, observations for the same country and year were combined by calculating the mean of the numerical indicators.

In [ ]:
years_countries2 = clean_data.groupby("Country")["Year"].nunique()
years_countries2.value_counts().sort_index()

In [ ]:
good_countries2 = years_countries2[years_countries2 == 10].index
clean_data = clean_data[clean_data["Country"].isin(good_countries2)]
clean_data.head()

Some countries have data for only a few years. To ensure more reliable analysis, only countries with observations available for all years in the dataset were retained.

In [ ]:
print(clean_data.shape)
display(clean_data.dtypes)

After cleaning and filtering, the dataset contains 716 rows and 11 columns. All remaining columns have appropriate data types for further analysis.

### Plots

In [ ]:
plt.figure(figsize=(10,6))
plt.hist(clean_data["Quality of Life Index"], bins = 20, edgecolor="black")

plt.title("Distribution of Quality of Life Index")
plt.xlabel("Quality of Life Index")
plt.ylabel("Frequency")

plt.grid(axis="y", linestyle="--")

plt.show()


Most observations have Quality of Life Index values between 90 and 190 points, while extremely low and extremely high values are relatively rare.

In [ ]:
plt.figure(figsize=(10,6))

plt.scatter(clean_data["Purchasing Power Index"], clean_data["Quality of Life Index"], alpha=0.6)

plt.title("Quality of Life vs Purchasing Power")
plt.xlabel("Purchasing Power Index")
plt.ylabel("Quality of Life Index")

plt.grid(alpha=0.3)

plt.show()

The scatter plot shows a positive relationship between Purchasing Power Index and Quality of Life Index. Countries with higher purchasing power generally tend to have higher quality-of-life scores. However, the spread of points indicates that purchasing power is not the only factor affecting quality of life, as countries with similar purchasing power may still have different quality-of-life levels.

### Merged Dataset

### Dataset Description

описание

In [ ]:
final_data = pd.merge(clean_data, all_data, on = ["Country", "Year"], how="inner")

In [ ]:
final_data.head()

In [ ]:
final_data.shape

In [ ]:
final_data.info()

In [ ]:
final_data.isna().sum()

In [ ]:
final_data.duplicated().sum()

In [ ]:
final_data["Year"].unique()

### Data cleanup

No additional data cleaning was required after merging the datasets. The merged dataset contains no missing values, duplicate rows, or incorrect data types.

### Detailed overview 1

In [ ]:
trend_economy = (
    final_data.groupby("Year")[[
        "Purchasing Power Index",
        "Cost of Living Index",
        "Property Price to Income Ratio"
    ]]
    .mean()
    .reset_index()
)

plt.figure(figsize=(9,6))

plt.plot(trend_economy["Year"], trend_economy["Purchasing Power Index"], marker="o", label="Purchasing Power Index", color = "#87CEFA")
plt.plot(trend_economy["Year"], trend_economy["Cost of Living Index"], marker="s", label="Cost of Living Index", color = "#FF69B4")
plt.plot(trend_economy["Year"], trend_economy["Property Price to Income Ratio"], marker="^", label="Property Price to Income Ratio", color = "#FFB6C1")

plt.title("Average Economic Indices by Year")
plt.xlabel("Year")
plt.ylabel("Average Index Value")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
first_year = final_data["Year"].min()
last_year = final_data["Year"].max()

data_first = final_data[final_data["Year"] == first_year]
data_last = final_data[final_data["Year"] == last_year]

top_first = data_first.nlargest(10, "Happiness Score")[["Country", "Happiness Score"]]
top_last = data_last.nlargest(10, "Happiness Score")[["Country", "Happiness Score"]]

first_countries = set(top_first["Country"])
last_countries = set(top_last["Country"])

# зелёный = страна есть в топ-10 в обоих годах
colors_first = ["green" if c in last_countries else "skyblue" for c in top_first["Country"]]
colors_last = ["green" if c in first_countries else "pink" for c in top_last["Country"]]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].barh(top_first["Country"], top_first["Happiness Score"], color=colors_first)
axes[0].invert_yaxis()
axes[0].set_title(f"Top-10 by Happiness ({first_year})\nGreen = also in top-10 in {last_year}")
axes[0].set_xlabel("Happiness Score")

axes[1].barh(top_last["Country"], top_last["Happiness Score"], color=colors_last)
axes[1].invert_yaxis()
axes[1].set_title(f"Top-10 by Happiness ({last_year})\nGreen = also in top-10 in {first_year}")
axes[1].set_xlabel("Happiness Score")

plt.suptitle(f"Comparison: Top-10 Happiest Countries, {first_year} vs {last_year}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
leaders = (
    final_data.loc[final_data.groupby("Year")["Happiness Score"].idxmax()]
    [["Year", "Country", "Happiness Score"]]
    .reset_index(drop=True)
)

plt.figure(figsize=(10,6))
bars = plt.bar(leaders["Year"].astype(str), leaders["Happiness Score"], color="#FF69B4")

for bar, country in zip(bars, leaders["Country"]):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             country, ha="center", va="bottom", rotation=45, fontsize=9)

plt.title("Happiness Leader by Year")
plt.xlabel("Year")
plt.ylabel("Happiness Score")
plt.ylim(0, leaders["Happiness Score"].max() + 1)
plt.tight_layout()
plt.show()

In [ ]:
top10_per_year = (
    final_data.groupby("Year")
    .apply(lambda x: x.nlargest(10, "Happiness Score")["Country"])
    .reset_index(drop=True)
)

top10_counts = top10_per_year.value_counts().head(10)

plt.figure(figsize=(10,6))
plt.barh(top10_counts.index, top10_counts.values, color="#FF69B4")
plt.gca().invert_yaxis()

plt.title("Countries Most Frequently in Top-10 by Happiness Score (All Years)")
plt.xlabel("Number of Years in Top-10")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

### Detailed overview 2

In [ ]:
median_happiness = final_data["Happiness Score"].median()

high_happiness = final_data[final_data["Happiness Score"] >= median_happiness]
low_happiness = final_data[final_data["Happiness Score"] < median_happiness]

comparison = pd.DataFrame({
    "High Happiness": [
        high_happiness["Quality of Life Index"].mean(),
        high_happiness["Purchasing Power Index"].mean(),
        high_happiness["Safety Index"].mean(),
        high_happiness["Pollution Index"].mean()
    ],

    "Low Happiness": [
        low_happiness["Quality of Life Index"].mean(),
        low_happiness["Purchasing Power Index"].mean(),
        low_happiness["Safety Index"].mean(),
        low_happiness["Pollution Index"].mean()
    ]
},

index=[
    "Quality of Life",
    "Purchasing Power",
    "Safety",
    "Pollution"
])

comparison = comparison.round(2)
display(comparison)

comparison.plot(
    kind="bar",
    figsize=(8,6)
)

plt.title("High vs Low Happiness Countries")
plt.xlabel("Indicator")
plt.ylabel("Mean Value")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)

plt.show()

In [ ]:
import seaborn as sns

correlation = final_data[["Happiness Score", "Family", "Freedom", "Generosity", "Quality of Life Index", "Purchasing Power Index", "Safety Index", "Health Care Index", "Cost of Living Index", "Property Price to Income Ratio", "Traffic Commute Time Index", "Pollution Index"]].corr()

plt.figure(figsize=(8,6))

sns.heatmap(
    correlation,
    annot=True,
    cmap="Blues",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

### Hypothesis check


### Hypothesis 1:

 What is the correlation between Purchasing Power Index and Happiness Score for countries with different levels of Freedom?

In [ ]:
final_data["Freedom Level"] = pd.cut(
    final_data["Freedom"],
    bins=3,
    labels=["Low Freedom", "Medium Freedom", "High Freedom"]
)

levels = ["Low Freedom", "Medium Freedom", "High Freedom"]
colors = ["#87CEFA", "#FF69B4", "#98FB98"]

print("Correlation between Purchasing Power Index and Happiness Score by Freedom Level:")
for level in levels:
    subset = final_data[final_data["Freedom Level"] == level]
    r = subset["Purchasing Power Index"].corr(subset["Happiness Score"])
    print(f"  {level}: r = {r:.3f}  (n = {len(subset)})")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, level, color in zip(axes, levels, colors):
    subset = final_data[final_data["Freedom Level"] == level]
    r = subset["Purchasing Power Index"].corr(subset["Happiness Score"])

    ax.scatter(subset["Purchasing Power Index"], subset["Happiness Score"],
               alpha=0.6, color=color, s=40)

    m, b = np.polyfit(subset["Purchasing Power Index"], subset["Happiness Score"], 1)
    x_sorted = sorted(subset["Purchasing Power Index"])
    ax.plot(x_sorted, [m*x + b for x in x_sorted], color="black", linewidth=1.5)

    ax.set_title(f"{level}\nr = {r:.2f}", fontsize=11)
    ax.set_xlabel("Purchasing Power Index")
    ax.set_ylabel("Happiness Score")
    ax.set_xlim(final_data["Purchasing Power Index"].min() - 5,
                final_data["Purchasing Power Index"].max() + 5)
    ax.set_ylim(final_data["Happiness Score"].min() - 0.2,
                final_data["Happiness Score"].max() + 0.2)
    ax.grid(alpha=0.3)

plt.suptitle("Hypothesis 1: The correlation between Purchasing Power and Happiness\nis stronger in countries with higher Freedom",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Hypothesis 2:

Does the relationship between Purchasing Power and Happiness differ for countries with different levels of Safety — and has this pattern changed over time (first half vs second half of the study period)?

In [ ]:
final_data["Safety Level"] = pd.cut(
    final_data["Safety Index"],
    bins=3,
    labels=["Low Safety", "Medium Safety", "High Safety"]
)

early = final_data[final_data["Year"] <= 2018]
late = final_data[final_data["Year"] > 2018]

periods = [("2015–2018", early), ("2019–2022", late)]
levels = ["Low Safety", "Medium Safety", "High Safety"]
colors = ["#FFB347", "#C39BD3", "#76D7C4"]

print("=== 2015–2018 ===")
for level in levels:
    s = early[early["Safety Level"] == level]
    r = s["Purchasing Power Index"].corr(s["Happiness Score"])
    print(f"  {level}: r = {r:.3f}, mean Happiness = {s['Happiness Score'].mean():.3f}, n = {len(s)}")

print("\n=== 2019–2022 ===")
for level in levels:
    s = late[late["Safety Level"] == level]
    r = s["Purchasing Power Index"].corr(s["Happiness Score"])
    print(f"  {level}: r = {r:.3f}, mean Happiness = {s['Happiness Score'].mean():.3f}, n = {len(s)}")

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for row, (period_label, period_data) in enumerate(periods):
    for col, (level, color) in enumerate(zip(levels, colors)):
        ax = axes[row][col]
        subset = period_data[period_data["Safety Level"] == level]

        ax.scatter(subset["Purchasing Power Index"], subset["Happiness Score"],
                   alpha=0.6, color=color, s=40)

        if len(subset) > 2:
            m, b = np.polyfit(subset["Purchasing Power Index"], subset["Happiness Score"], 1)
            x_sorted = sorted(subset["Purchasing Power Index"])
            ax.plot(x_sorted, [m*x + b for x in x_sorted], color="black", linewidth=1.5)
            r = subset["Purchasing Power Index"].corr(subset["Happiness Score"])
            ax.set_title(f"{level}\n{period_label}  |  r = {r:.2f}", fontsize=11)
        else:
            ax.set_title(f"{level}\n{period_label}  |  n too small", fontsize=11)

        ax.set_xlabel("Purchasing Power Index")
        ax.set_ylabel("Happiness Score")
        ax.set_xlim(final_data["Purchasing Power Index"].min() - 5,
                    final_data["Purchasing Power Index"].max() + 5)
        ax.set_ylim(final_data["Happiness Score"].min() - 0.2,
                    final_data["Happiness Score"].max() + 0.2)
        ax.grid(alpha=0.3)

plt.suptitle("Hypothesis 2: The impact of Purchasing Power on Happiness\ndiffers by Safety Level and changes over time",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


### Hypothesis 3:

The three factors most strongly correlated with the Quality of Life Index are also the three factors most strongly correlated with the Happiness Score.

In [ ]:
cols = ["Family", "Freedom", "Generosity",
        "Purchasing Power Index", "Safety Index",
        "Health Care Index", "Cost of Living Index",
        "Property Price to Income Ratio",
        "Traffic Commute Time Index", "Pollution Index"]

corr_qol = (final_data[cols + ["Quality of Life Index"]]
            .corr()["Quality of Life Index"]
            .drop("Quality of Life Index")
            .abs()
            .sort_values(ascending=False))

corr_hap = (final_data[cols + ["Happiness Score"]]
            .corr()["Happiness Score"]
            .drop("Happiness Score")
            .abs()
            .sort_values(ascending=False))

top3_qol = list(corr_qol.head(3).index)
top3_hap = list(corr_hap.head(3).index)
overlap = set(top3_qol) & set(top3_hap)

print("Top-3 correlated with Quality of Life Index:", top3_qol)
print("Top-3 correlated with Happiness Score:", top3_hap)
print("Overlap:", overlap)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

bars0 = axes[0].bar(corr_qol.head(5).index, corr_qol.head(5).values,
                     color=["#E74C3C" if f in overlap else "#F1948A" for f in corr_qol.head(5).index])
axes[0].set_title("Top-5 Factors Correlated\nwith Quality of Life Index", fontsize=11)
axes[0].set_ylabel("Absolute Correlation")
axes[0].tick_params(axis="x", rotation=45)
axes[0].set_ylim(0, 1)
axes[0].grid(axis="y", alpha=0.3)

for bar, val in zip(bars0, corr_qol.head(5).values):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.01,
                 f"{val:.2f}", ha="center", va="bottom", fontsize=9)

bars1 = axes[1].bar(corr_hap.head(5).index, corr_hap.head(5).values,
                     color=["#8E44AD" if f in overlap else "#C39BD3" for f in corr_hap.head(5).index])
axes[1].set_title("Top-5 Factors Correlated\nwith Happiness Score", fontsize=11)
axes[1].set_ylabel("Absolute Correlation")
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_ylim(0, 1)
axes[1].grid(axis="y", alpha=0.3)

for bar, val in zip(bars1, corr_hap.head(5).values):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.01,
                 f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Hypothesis 3: Do the same factors drive Quality of Life and Happiness?\n(darker bars = factor appears in top-3 of both indices)",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### Data transformation

### Discussion

TEST